<a href="https://colab.research.google.com/github/leadministrador/mineria-datos-ii-proyecto-integrador/blob/main/notebooks/02_streaming_bronze.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [52]:
!pip -q install pyspark

In [53]:
!unzip -q -o cloud_provider_challenge_dataset_v1.zip -d /content/proyecto

In [54]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("proyecto-integrador")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Versión de Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Versión de Spark: 4.0.5
Master: local[*]
Shuffle partitions: 8


In [55]:
LANDING = "/content/proyecto/datalake/landing"
eventos = spark.read.json(LANDING + "/usage_events_stream")
eventos.printSchema()
print("Eventos:", eventos.count())

root
 |-- carbon_kg: double (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- event_id: string (nullable = true)
 |-- genai_tokens: long (nullable = true)
 |-- metric: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- schema_version: long (nullable = true)
 |-- service: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- value: string (nullable = true)

Eventos: 43200


In [56]:
esquema = ("event_id STRING, timestamp STRING, org_id STRING, resource_id STRING, "
           "service STRING, region STRING, metric STRING, value STRING, unit STRING, "
           "cost_usd_increment DECIMAL(10,4), schema_version BIGINT, "
           "carbon_kg DECIMAL(10,4), genai_tokens BIGINT")
print("Esquema listo")

Esquema listo


In [57]:
BRONZE = "/content/proyecto/datalake/bronze/usage_events"
CHECK = "/content/proyecto/checkpoints/bronze_usage_events"

def ingesta():
    eventos_stream = (spark.readStream
        .schema(esquema)
        .option("maxFilesPerTrigger", 10)
        .json(LANDING + "/usage_events_stream")
        .withColumn("event_ts", F.col("timestamp").cast("timestamp"))
        .withColumn("event_date", F.to_date("event_ts"))
        .withColumn("ingest_ts", F.current_timestamp())
        .withColumn("source_file", F.col("_metadata.file_name"))
        .withWatermark("event_ts", "60 days")
        .dropDuplicatesWithinWatermark(["event_id"]))
    q = (eventos_stream.writeStream
        .format("parquet")
        .option("path", BRONZE)
        .option("checkpointLocation", CHECK)
        .partitionBy("event_date")
        .outputMode("append")
        .trigger(availableNow=True)
        .start())
    q.awaitTermination()

ingesta()
print("Filas en Bronze:", spark.read.parquet(BRONZE).count())

Filas en Bronze: 43200


In [58]:
antes = spark.read.parquet(BRONZE).count()
ingesta()
despues = spark.read.parquet(BRONZE).count()
print("Antes:", antes, "· Después:", despues)

spark.read.parquet(BRONZE).select("event_id", "event_date", "source_file", "ingest_ts").show(3, truncate=False)

Antes: 43200 · Después: 43200
+----------------+----------+----------------------+----------------------+
|event_id        |event_date|source_file           |ingest_ts             |
+----------------+----------+----------------------+----------------------+
|evt_bwmz1gmz99c7|2025-07-28|events_part_0012.jsonl|2026-11-17 03:45:39.63|
|evt_452y9czirbke|2025-07-28|events_part_0116.jsonl|2026-11-17 03:45:39.63|
|evt_qa96t5hbb9j0|2025-07-28|events_part_0116.jsonl|2026-11-17 03:45:39.63|
+----------------+----------+----------------------+----------------------+
only showing top 3 rows


In [59]:
SILVER = "/content/proyecto/datalake/silver/usage_events"
QUARANTINE = "/content/proyecto/datalake/quarantine/usage_events"

esquema_bronze = esquema + ", event_ts TIMESTAMP, event_date DATE, ingest_ts TIMESTAMP, source_file STRING"

RULE = ("coalesce(event_id IS NOT NULL AND cost_usd_increment >= -0.01 "
        "AND (value IS NULL OR unit IS NOT NULL), false)")

orgs = (spark.read.option("header", True).option("inferSchema", True)
        .csv(LANDING + "/customers_orgs.csv")
        .select("org_id", "industry", "plan_tier"))

bronze_stream = spark.readStream.schema(esquema_bronze).parquet(BRONZE)

silver = (bronze_stream.filter(RULE)
          .withColumn("value", F.col("value").cast("decimal(12,4)"))
          .join(F.broadcast(orgs), "org_id", "left"))
quarantine = bronze_stream.filter(f"NOT {RULE}")

(silver.writeStream.format("parquet")
    .option("path", SILVER)
    .option("checkpointLocation", "/content/proyecto/checkpoints/silver_usage_events")
    .partitionBy("event_date")
    .trigger(availableNow=True).start().awaitTermination())

(quarantine.writeStream.format("parquet")
    .option("path", QUARANTINE)
    .option("checkpointLocation", "/content/proyecto/checkpoints/quarantine_usage_events")
    .trigger(availableNow=True).start().awaitTermination())

print("Silver:", spark.read.parquet(SILVER).count())
print("Quarantine:", spark.read.parquet(QUARANTINE).count())

Silver: 40956
Quarantine: 2244


In [60]:
for tabla in ["customers_orgs", "users", "billing_monthly"]:
    df = (spark.read.option("header", True).option("inferSchema", True)
          .csv(LANDING + "/" + tabla + ".csv")
          .withColumn("ingest_ts", F.current_timestamp())
          .withColumn("source_file", F.col("_metadata.file_name")))
    df.write.mode("overwrite").parquet("/content/proyecto/datalake/bronze/" + tabla)
    print(tabla, spark.read.parquet("/content/proyecto/datalake/bronze/" + tabla).count())

customers_orgs 80
users 800
billing_monthly 240


In [61]:
def leer(tabla, esquema_m, clave):
    return (spark.read.option("header", True).schema(esquema_m)
            .csv(LANDING + "/" + tabla + ".csv")
            .dropDuplicates([clave])
            .withColumn("ingest_ts", F.current_timestamp())
            .withColumn("source_file", F.col("_metadata.file_name")))

BRONZE_M = "/content/proyecto/datalake/bronze/"

In [62]:
leer("customers_orgs",
     "org_id STRING, org_name STRING, industry STRING, hq_region STRING, "
     "plan_tier STRING, is_enterprise STRING, signup_date DATE, sales_rep STRING, "
     "lifecycle_stage STRING, marketing_source STRING, nps_score DECIMAL(5,1)",
     "org_id").write.mode("overwrite").parquet(BRONZE_M + "customers_orgs")

print("customers_orgs", spark.read.parquet(BRONZE_M + "customers_orgs").count())

customers_orgs 80


In [63]:
leer("users",
     "user_id STRING, org_id STRING, email STRING, role STRING, active STRING, "
     "created_at DATE, last_login DATE",
     "user_id").write.mode("overwrite").parquet(BRONZE_M + "users")

print("users", spark.read.parquet(BRONZE_M + "users").count())

users 800


In [64]:
leer("billing_monthly",
     "invoice_id STRING, org_id STRING, month DATE, subtotal DECIMAL(12,2), "
     "credits DECIMAL(12,2), taxes DECIMAL(12,2), currency STRING, "
     "exchange_rate_to_usd DECIMAL(10,5)",
     "invoice_id").write.mode("overwrite").partitionBy("month").parquet(BRONZE_M + "billing_monthly")

print("billing_monthly", spark.read.parquet(BRONZE_M + "billing_monthly").count())

billing_monthly 240


In [65]:
SILVER_B = "/content/proyecto/datalake/silver/billing_monthly"

silver_billing = (spark.read.parquet(BRONZE_M + "billing_monthly")
    .withColumn("credits", F.expr("coalesce(credits, 0)")))

silver_billing.write.mode("overwrite").partitionBy("month").parquet(SILVER_B)

print("Silver billing:", spark.read.parquet(SILVER_B).count())

Silver billing: 240


In [66]:
GOLD = "/content/proyecto/datalake/gold/org_daily_usage_by_service"
silver_ev = spark.read.parquet(SILVER)
claves = ["org_id", F.col("event_date").alias("usage_date"), "service"]

costos = (silver_ev.groupBy(claves)
          .agg(F.round(F.sum("cost_usd_increment"), 4).alias("daily_cost_usd"),
               F.sum("genai_tokens").alias("genai_tokens"),
               F.round(F.sum("carbon_kg"), 4).alias("carbon_kg")))

requests = (silver_ev.filter("metric = 'requests'")
            .groupBy(claves)
            .agg(F.sum("value").alias("requests")))

gold = costos.join(requests, ["org_id", "usage_date", "service"], "left")
gold.write.mode("overwrite").partitionBy("usage_date").parquet(GOLD)

print("Gold filas:", spark.read.parquet(GOLD).count())
spark.read.parquet(GOLD).show(5)

Gold filas: 11050
+------------+----------+--------------+------------+---------+--------+----------+
|      org_id|   service|daily_cost_usd|genai_tokens|carbon_kg|requests|usage_date|
+------------+----------+--------------+------------+---------+--------+----------+
|org_ykl1cq99|networking|        1.5237|        NULL|   0.0270|124.0000|2025-08-28|
|org_uzfk8ut0|   compute|       30.8312|        NULL|   0.0716|337.0000|2025-08-28|
|org_l58kxo9t|   storage|        2.8733|        NULL|   0.0253|112.0000|2025-08-28|
|org_fel6246h|     genai|       11.8753|        1298|   0.0250|111.0000|2025-08-28|
|org_chj755nf|   compute|        4.4334|        NULL|   0.0112|    NULL|2025-08-28|
+------------+----------+--------------+------------+---------+--------+----------+
only showing top 5 rows


In [67]:
!rm -rf /content/proyecto/datalake/quarantine/usage_events /content/proyecto/checkpoints/quarantine_usage_events

In [68]:
quarantine = (bronze_stream.filter(f"NOT {RULE}")
    .withColumn("motivo",
        F.when(F.col("event_id").isNull(), "sin_event_id")
         .when(F.col("cost_usd_increment") < -0.01, "costo_negativo")
         .otherwise("unit_vacio")))

q = (quarantine.writeStream.queryName("quarantine_usage_events")
    .format("parquet")
    .option("path", QUARANTINE)
    .option("checkpointLocation", "/content/proyecto/checkpoints/quarantine_usage_events")
    .trigger(availableNow=True).start())
q.awaitTermination()

print(q.name, q.lastProgress["numInputRows"])
spark.read.parquet(QUARANTINE).groupBy("motivo").count().show()

quarantine_usage_events 43200
+--------------+-----+
|        motivo|count|
+--------------+-----+
|costo_negativo|  211|
|    unit_vacio| 2033|
+--------------+-----+



In [69]:
spark.read.parquet(QUARANTINE).select("event_id", "cost_usd_increment", "value", "unit", "motivo").show(6, truncate=False)

+----------------+------------------+-------+-----+--------------+
|event_id        |cost_usd_increment|value  |unit |motivo        |
+----------------+------------------+-------+-----+--------------+
|evt_iqus3t7c69bf|16.6161           |137.0  |NULL |unit_vacio    |
|evt_5grn5yunlayq|0.5208            |3.218  |NULL |unit_vacio    |
|evt_5q8ivd827y1u|0.1764            |15.8957|NULL |unit_vacio    |
|evt_j8qdm330floc|5.2298            |111.0  |NULL |unit_vacio    |
|evt_fqng89vb92g1|-6.1267           |108.0  |count|costo_negativo|
|evt_cksl5d00a1lu|3.0918            |140.0  |NULL |unit_vacio    |
+----------------+------------------+-------+-----+--------------+
only showing top 6 rows


In [70]:
spark.read.parquet(QUARANTINE).filter("motivo = 'costo_negativo'").select("event_id", "cost_usd_increment", "value", "unit", "motivo").show(3, truncate=False)

+----------------+------------------+-------+--------+--------------+
|event_id        |cost_usd_increment|value  |unit    |motivo        |
+----------------+------------------+-------+--------+--------------+
|evt_fqng89vb92g1|-6.1267           |108.0  |count   |costo_negativo|
|evt_kwcwbndmwq2t|-0.7245           |20.1209|gb_hours|costo_negativo|
|evt_y9wmys4c3k5n|-11.3120          |119.0  |count   |costo_negativo|
+----------------+------------------+-------+--------+--------------+
only showing top 3 rows


In [71]:
!pip -q install cassandra-driver

In [72]:
import os
from google.colab import userdata
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider

os.environ["ASTRA_TOKEN"] = userdata.get("ASTRA_TOKEN")
os.environ["ASTRA_SCB"] = "/content/secure-connect-mineria-datos-ii.zip"

cluster = Cluster(
    cloud={"secure_connect_bundle": os.environ["ASTRA_SCB"]},
    auth_provider=PlainTextAuthProvider("token",
                                        os.environ["ASTRA_TOKEN"]))
session = cluster.connect("proyecto_integrador")

fila = session.execute("SELECT release_version FROM system.local").one()
print("Conectado. Versión CQL del servicio:", fila.release_version)

Conectado. Versión CQL del servicio: 4.0.11.0-6819aa9c7f6c


In [77]:
from cassandra.concurrent import execute_concurrent_with_args as run

gold_cass = spark.read.parquet(GOLD).select(
    "org_id", "service",
    F.col("usage_date").cast("date"),
    F.col("daily_cost_usd").cast("double"),
    F.col("requests").cast("double"),
    F.col("genai_tokens").cast("double"),
    F.col("carbon_kg").cast("double"))

filas = [tuple(r) for r in gold_cass.collect()]

ins1 = session.prepare("INSERT INTO org_daily_usage_by_service (org_id, service, usage_date,"
                       " daily_cost_usd, requests, genai_tokens, carbon_kg) VALUES (?, ?, ?, ?, ?, ?, ?)")
run(session, ins1, filas, concurrency=50)
print("Filas cargadas:", len(filas))

Filas cargadas: 11050


In [75]:
top14 = (spark.read.parquet(GOLD)
         .filter("usage_date >= '2025-08-18'")
         .groupBy("org_id", "service")
         .agg(F.round(F.sum("daily_cost_usd"), 4).cast("double").alias("cost_14d_usd"))
         .select("org_id", "cost_14d_usd", "service"))

filas2 = [tuple(r) for r in top14.collect()]

ins2 = session.prepare("INSERT INTO top_services_14d_by_org (org_id, cost_14d_usd, service)"
                       " VALUES (?, ?, ?)")
run(session, ins2, filas2, concurrency=50)
print("Filas cargadas:", len(filas2))

Filas cargadas: 262


In [76]:
org, costo, srv = filas2[0]
print("Organización:", org, "· Servicio:", srv)

print("\nConsulta 1: costos y requests diarios por organización y servicio, 18/08 al 31/08")
q1 = session.execute("SELECT usage_date, daily_cost_usd, requests FROM org_daily_usage_by_service"
                     " WHERE org_id = %s AND service = %s AND usage_date >= %s AND usage_date <= %s",
                     (org, srv, "2025-08-18", "2025-08-31"))
for r in q1:
    print(r.usage_date, r.daily_cost_usd, r.requests)

print("\nConsulta 2: Top 3 servicios por costo de los últimos 14 días")
q2 = session.execute("SELECT service, cost_14d_usd FROM top_services_14d_by_org"
                     " WHERE org_id = %s LIMIT 3", (org,))
for r in q2:
    print(r.service, r.cost_14d_usd)

Organización: org_g8sbi4q2 · Servicio: database

Consulta 1: costos y requests diarios por organización y servicio, 18/08 al 31/08
2025-08-31 4.6026 108.0
2025-08-29 6.8558 126.0
2025-08-28 10.8956 212.0
2025-08-25 7.3396 115.0
2025-08-24 0.7414 None
2025-08-23 15.5001 263.0
2025-08-22 0.7607 None
2025-08-21 12.321 250.0
2025-08-20 13.4362 264.0
2025-08-19 11.5807 246.0
2025-08-18 8.0045 118.0

Consulta 2: Top 3 servicios por costo de los últimos 14 días
database 92.0382
compute 84.1779
storage 62.054
